# M6 PTCST training (method chính)
Train the locked five-seed PTCST sweep first. Checkpoints are synced to Drive and resumed automatically after a runtime interruption. Deep baselines are optional and run after the proposed method.

In [ ]:
# Bootstrap this notebook even when it is opened in a fresh Colab kernel.
from pathlib import Path
import urllib.request
_BOOTSTRAP_REPO = Path('/content/kltn')
_BOOTSTRAP_SCRIPT = _BOOTSTRAP_REPO / 'scripts' / 'colab_bootstrap.py'
if not _BOOTSTRAP_SCRIPT.exists():
    raw = 'https://raw.githubusercontent.com/maiphuowng205/kltn/5ebae86/scripts/colab_bootstrap.py'
    urllib.request.urlretrieve(raw, '/content/colab_bootstrap.py')
    _BOOTSTRAP_SCRIPT = Path('/content/colab_bootstrap.py')
exec(_BOOTSTRAP_SCRIPT.read_text(encoding='utf-8'), globals())


In [ ]:
# Main method: locked PTCST five-seed sweep.
import subprocess, sys
ptcst_run = WORKSPACE / 'runs' / 'v3_ptcst_seed_sweep'
ptcst_sync = DRIVE_RUN_ROOT / 'checkpoints' / 'v3_ptcst_seed_sweep'
def run_stage(label, script_name, extra_args):
    script_path = Path(script_name)
    if not script_path.is_absolute(): script_path = REPO / 'scripts' / script_path
    cmd = [sys.executable, str(script_path), *extra_args]
    print(f'\n=== {label} ===')
    print(' '.join(map(str, cmd)))
    result = subprocess.run(cmd, check=False, capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print('STDERR:\n' + result.stderr)
    print(f'[{label}] returncode={result.returncode}')
    if result.returncode != 0:
        detail = (result.stderr or result.stdout or 'no subprocess output').strip()
        raise RuntimeError(f'{label} failed with returncode {result.returncode}:\n{detail}')
    return result
sweep_script = Path('/content/run_v3_ptcst_seed_sweep_fixed.py')
if not sweep_script.exists():
    raw = 'https://raw.githubusercontent.com/maiphuowng205/kltn/aabba79/scripts/run_v3_ptcst_seed_sweep.py'
    urllib.request.urlretrieve(raw, sweep_script)
seed_names = ['7', '19', '43', '71', '101']
required_names = ['metrics.json', 'config.yaml', 'run_manifest.json', 'best.pt']
seed_complete = all(all((ptcst_run / f'seed_{seed}' / name).exists() for name in required_names) for seed in seed_names)
if not seed_complete:
    run_stage('PTCST seed sweep', str(sweep_script), ['--data-root', str(DATA_ROOT), '--run-root', str(ptcst_run), '--seeds', *seed_names, '--epochs', '100', '--checkpoint-sync-root', str(ptcst_sync)])
else:
    print('Existing completed PTCST sweep found; no retraining needed:', ptcst_run)
import pandas as pd
seed_summary = pd.read_parquet(ptcst_run / 'seed_summary.parquet')
display(seed_summary)
assert len(seed_summary) == 5, 'Expected all five locked PTCST seeds.'


In [ ]:
# Persist the completed method artifacts to Drive.
from shutil import copytree
drive_runs = Path('/content/drive/MyDrive/kltn/runs')
drive_runs.mkdir(parents=True, exist_ok=True)
copytree(ptcst_run, drive_runs / 'v3_ptcst_seed_sweep', dirs_exist_ok=True)
print('PTCST artifacts synced to:', drive_runs / 'v3_ptcst_seed_sweep')
print('Checkpoints synced to:', ptcst_sync)


In [ ]:
# Optional: run vanilla deep baselines after the proposed method.
RUN_DEEP_BASELINES = False
deep_run = WORKSPACE / 'runs' / 'v3_deep_baselines'
deep_sync = DRIVE_RUN_ROOT / 'checkpoints' / 'v3_deep_baselines'
if RUN_DEEP_BASELINES:
    run_stage('deep baselines', 'run_v3_deep_baseline_sweep.py', ['--data-root', str(DATA_ROOT), '--run-root', str(deep_run), '--checkpoint-sync-root', str(deep_sync), '--epochs', '100'])
    copytree(deep_run, drive_runs / 'v3_deep_baselines', dirs_exist_ok=True)
    print('Deep baseline artifacts synced to:', drive_runs / 'v3_deep_baselines')
else:
    print('Deep baselines skipped. Set RUN_DEEP_BASELINES = True only after PTCST is complete.')
